# Visualize the OC + HTN gene network

Two outputs:
1. **Cytoscape files** (`cytoscape_edges.csv`, `cytoscape_nodes.csv`) for publication figures.
2. **Interactive HTML views** (vis-network, no extra install needed) to explore in the browser: zoom, drag, hover a gene for details.

Colours: **orange** = ovarian cancer only, **blue** = hypertension only, **aqua** = shared by both.

In [1]:
import json
from pathlib import Path

import networkx as nx
import pandas as pd

CUTOFF = 0.4
DISEASE_A, DISEASE_B = "OC", "HTN"

INTERACTOME_FILE = "../hypertention - breast cancer/outputs/interactome/interactome_both_symbols.csv"
OUT_DIR = Path("outputs") / f"cutoff_{CUTOFF}" / f"network_{DISEASE_A}_{DISEASE_B}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DISEASE_FILES = {
    "OC": "inputs/ovarian cancer-MONDO_0008170.tsv",
    "HTN": "inputs/hypertensive disorder-MONDO_0005044.tsv",
    "SBP": "inputs/systolic blood pressure-EFO_0006335.tsv",
}

# fixed colour per group (validated colour-blind-safe set)
GROUP_COLORS = {
    f"{DISEASE_A} only": "#eb6834",  # orange
    f"{DISEASE_B} only": "#2a78d6",  # blue
    "shared": "#1baf7a",  # aqua
}

## Load the network and the two disease gene sets

In [2]:
interactome_df = pd.read_csv(INTERACTOME_FILE)
G = nx.from_pandas_edgelist(interactome_df, "symbol_1", "symbol_2", edge_attr="data_source")

scores = {}
genes = {}
for name in (DISEASE_A, DISEASE_B):
    df = pd.read_csv(DISEASE_FILES[name], sep="\t", low_memory=False)
    df = df[df["globalScore"] >= CUTOFF]
    scores[name] = df.set_index("symbol")["globalScore"]
    genes[name] = set(df["symbol"]) & set(G.nodes())

A, B = genes[DISEASE_A], genes[DISEASE_B]
print(f"{DISEASE_A}: {len(A)} genes, {DISEASE_B}: {len(B)} genes, shared: {len(A & B)}")

OC: 403 genes, HTN: 308 genes, shared: 26


## Build the subnetwork
Keep only the disease genes and the links **between them**. Genes with no link to any other
disease gene are dropped from the picture (they would just float around on their own).

In [3]:
sub = G.subgraph(A | B).copy()
isolated = [n for n in sub if sub.degree(n) == 0]
sub.remove_nodes_from(isolated)


def group_of(gene):
    if gene in A and gene in B:
        return "shared"
    return f"{DISEASE_A} only" if gene in A else f"{DISEASE_B} only"


print(f"Subnetwork: {sub.number_of_nodes()} genes, {sub.number_of_edges()} links")
print(f"Dropped {len(isolated)} genes with no link to another disease gene")
pd.Series({n: group_of(n) for n in sub}).value_counts()

Subnetwork: 536 genes, 2261 links
Dropped 149 genes with no link to another disease gene


OC only     336
HTN only    177
shared       23
Name: count, dtype: int64

## Count links between the groups
How many links stay inside one disease and how many cross between the two diseases?

In [4]:
link_types = pd.Series(
    [" — ".join(sorted((group_of(u), group_of(v)))) for u, v in sub.edges()]
).value_counts()
link_types

OC only — OC only      1431
HTN only — OC only      480
OC only — shared        189
HTN only — HTN only     113
HTN only — shared        36
shared — shared          12
Name: count, dtype: int64

## 1. Export for Cytoscape

**How to open in Cytoscape**
1. *File → Import → Network from File* → choose `cytoscape_edges.csv`
   (source = `source`, target = `target`).
2. *File → Import → Table from File* → choose `cytoscape_nodes.csv`, import as **node table**, key column `gene`.
3. *Style* panel: **Fill Color** → column `group` → *Discrete mapping* → paste the colours above;
   **Size** → column `degree_in_subnetwork` → *Continuous mapping*; **Label** → `gene`.
4. *Layout → Prefuse Force Directed* (or *yFiles Organic*).
5. *File → Export → Network to Image* (PDF/SVG for the paper).

In [5]:
edges_out = pd.DataFrame(
    [(u, v, d["data_source"]) for u, v, d in sub.edges(data=True)],
    columns=["source", "target", "data_source"],
)
nodes_out = pd.DataFrame(
    {
        "gene": list(sub.nodes()),
        "group": [group_of(n) for n in sub],
        f"score_{DISEASE_A}": [scores[DISEASE_A].get(n) for n in sub],
        f"score_{DISEASE_B}": [scores[DISEASE_B].get(n) for n in sub],
        "degree_in_subnetwork": [sub.degree(n) for n in sub],
        "degree_in_full_network": [G.degree(n) for n in sub],
    }
)

edges_out.to_csv(OUT_DIR / "cytoscape_edges.csv", index=False)
nodes_out.to_csv(OUT_DIR / "cytoscape_nodes.csv", index=False)
print(f"Saved {OUT_DIR / 'cytoscape_edges.csv'} ({len(edges_out)} links)")
print(f"Saved {OUT_DIR / 'cytoscape_nodes.csv'} ({len(nodes_out)} genes)")

Saved outputs/cutoff_0.4/network_OC_HTN/cytoscape_edges.csv (2261 links)
Saved outputs/cutoff_0.4/network_OC_HTN/cytoscape_nodes.csv (536 genes)


## 2. Interactive HTML views
Open the `.html` files in a browser (needs internet once, to load the vis-network library). Hover a gene to see its group, scores and number of links.
Bigger circle = more links inside this subnetwork. Only shared genes get a text label, to keep it readable.

In [6]:
HTML_TEMPLATE = """<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>__TITLE__</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/vis-network/9.1.9/standalone/umd/vis-network.min.js"></script>
<style>
  body { margin: 0; font-family: sans-serif; background: #fcfcfb; color: #0b0b0b; }
  #legend { position: absolute; top: 12px; left: 12px; background: #ffffffdd; padding: 8px 12px;
            border: 1px solid #e0dfd9; border-radius: 6px; font-size: 14px; z-index: 1; }
  #legend span { display: inline-block; width: 12px; height: 12px; border-radius: 50%; margin-right: 6px; vertical-align: -1px; }
  #net { width: 100vw; height: 100vh; }
  div.vis-tooltip { white-space: pre-line !important; font-family: sans-serif; }
</style>
</head>
<body>
<div id="legend"><b>__TITLE__</b><br>__LEGEND__<br><small>__COUNTS__ · hover a gene for details</small></div>
<div id="net"></div>
<script>
  const nodes = new vis.DataSet(__NODES__);
  const edges = new vis.DataSet(__EDGES__);
  const network = new vis.Network(document.getElementById("net"), {nodes, edges}, {
    nodes: {shape: "dot", borderWidth: 2, font: {size: 22, color: "#0b0b0b", strokeWidth: 3, strokeColor: "#fcfcfb"}},
    edges: {color: {color: "#c9c8c2", highlight: "#52514e"}, width: 1, smooth: false},
    physics: {solver: "barnesHut", barnesHut: {gravitationalConstant: -40000, centralGravity: 0.4,
                                               springLength: 250, springConstant: 0.02, avoidOverlap: 1},
              stabilization: {iterations: 1000}},
    interaction: {hover: true, tooltipDelay: 100}
  });
  // once the layout has settled: stop the physics (nodes stay where they are) and zoom to fit
  network.once("stabilizationIterationsDone", () => {
    network.setOptions({physics: false});
    network.fit();
  });
</script>
</body>
</html>"""


def draw(graph, path, title, label_groups=("shared",)):
    nodes = []
    for n in graph.nodes():
        g = group_of(n)
        lines = [n, f"group: {g}"]
        for d in (DISEASE_A, DISEASE_B):
            s = scores[d].get(n)
            if s is not None:
                lines.append(f"{d} score: {s:.2f}")
        lines.append(f"links here: {graph.degree(n)} | links in full network: {G.degree(n)}")
        nodes.append(
            {
                "id": n,
                "label": n if g in label_groups else "",
                "title": "\n".join(lines),
                "color": {"background": GROUP_COLORS[g], "border": "#fcfcfb"},
                "size": 8 + 4 * graph.degree(n) ** 0.5,
            }
        )
    edges = [{"from": u, "to": v} for u, v in graph.edges()]

    counts = pd.Series([group_of(n) for n in graph]).value_counts()
    legend = "<br>".join(
        f'<span style="background:{c}"></span>{g} ({counts.get(g, 0)})'
        for g, c in GROUP_COLORS.items()
    )
    html = (
        HTML_TEMPLATE.replace("__TITLE__", title)
        .replace("__LEGEND__", legend)
        .replace("__COUNTS__", f"{graph.number_of_nodes()} genes, {graph.number_of_edges()} links")
        .replace("__NODES__", json.dumps(nodes))
        .replace("__EDGES__", json.dumps(edges))
    )
    with open(path, "w") as f:
        f.write(html)
    print(f"Saved {path}: {graph.number_of_nodes()} genes, {graph.number_of_edges()} links")


# View 1: the whole OC + HTN subnetwork
draw(sub, OUT_DIR / "all_genes.html", f"{DISEASE_A} + {DISEASE_B} disease genes")

Saved outputs/cutoff_0.4/network_OC_HTN/all_genes.html: 536 genes, 2261 links


### View 2: shared genes and their direct disease-gene neighbours
A smaller, clearer picture: every shared gene plus the OC / HTN genes linked directly to it.
This shows how the shared genes act as **bridges** between the two diseases.

In [7]:
shared = {n for n in sub if group_of(n) == "shared"}
neighbours = {m for n in shared for m in sub.neighbors(n)}
bridge = sub.subgraph(shared | neighbours).copy()

draw(
    bridge,
    OUT_DIR / "shared_bridges.html",
    "Shared genes and their neighbours",
    label_groups=("shared", f"{DISEASE_B} only"),
)

Saved outputs/cutoff_0.4/network_OC_HTN/shared_bridges.html: 138 genes, 787 links


### View 3: only the links that cross between the two diseases
Genes of one disease that directly touch genes of the other disease.

In [8]:
cross_edges = [
    (u, v) for u, v in sub.edges() if group_of(u) != group_of(v) or group_of(u) == "shared"
]
cross = sub.edge_subgraph(cross_edges).copy()

draw(
    cross,
    OUT_DIR / "cross_links.html",
    f"Links crossing {DISEASE_A} and {DISEASE_B}",
    label_groups=("shared", f"{DISEASE_A} only", f"{DISEASE_B} only"),
)

Saved outputs/cutoff_0.4/network_OC_HTN/cross_links.html: 362 genes, 717 links


### Show a view inside the notebook (optional)

In [9]:
from IPython.display import IFrame

IFrame(str(OUT_DIR / "shared_bridges.html"), width="100%", height=870)

---
# Three diseases together: OC, HTN and SBP

Every gene falls into one of 7 groups (OC only, HTN only, SBP only, OC + HTN, OC + SBP, HTN + SBP, all three).
Seven colours can't be told apart reliably, so we use only the **three disease colours** and draw each gene as a
small **pie**: a gene in one disease is a plain circle, a gene in two diseases is half/half, a gene in all three has three slices.

In the HTML view, **click a group in the legend to hide/show it**. Genes in 2+ diseases always have a text label.

In [10]:
import math
from urllib.parse import quote

DISEASES = ["OC", "HTN", "SBP"]
OUT_DIR_3 = Path("outputs") / f"cutoff_{CUTOFF}" / "network_OC_HTN_SBP"
OUT_DIR_3.mkdir(parents=True, exist_ok=True)

# one colour per disease (validated colour-blind-safe for 3 colours)
DISEASE_COLORS = {"OC": "#eb6834", "HTN": "#2a78d6", "SBP": "#1baf7a"}

scores3 = {}
genes3 = {}
for name in DISEASES:
    df = pd.read_csv(DISEASE_FILES[name], sep="\t", low_memory=False)
    df = df[df["globalScore"] >= CUTOFF]
    scores3[name] = df.set_index("symbol")["globalScore"]
    genes3[name] = set(df["symbol"]) & set(G.nodes())


def diseases_of(gene):
    """Which of the three diseases this gene belongs to, e.g. ('OC', 'SBP')."""
    return tuple(d for d in DISEASES if gene in genes3[d])


def group_name(gene):
    ds = diseases_of(gene)
    return f"{ds[0]} only" if len(ds) == 1 else " + ".join(ds)


# fixed order of the 7 groups (used for the legend)
GROUPS_3 = [
    "OC only",
    "HTN only",
    "SBP only",
    "OC + HTN",
    "OC + SBP",
    "HTN + SBP",
    "OC + HTN + SBP",
]

sub3 = G.subgraph(set().union(*genes3.values())).copy()
isolated3 = [n for n in sub3 if sub3.degree(n) == 0]
sub3.remove_nodes_from(isolated3)

print(
    f"Subnetwork: {sub3.number_of_nodes()} genes, {sub3.number_of_edges()} links "
    f"(dropped {len(isolated3)} genes with no link to another disease gene)"
)
pd.Series([group_name(n) for n in sub3]).value_counts().reindex(GROUPS_3, fill_value=0)

Subnetwork: 741 genes, 3056 links (dropped 232 genes with no link to another disease gene)


OC only           318
HTN only           77
SBP only          185
OC + HTN            9
OC + SBP           24
HTN + SBP         114
OC + HTN + SBP     14
Name: count, dtype: int64

## Export for Cytoscape
The node table has one yes/no column per disease (`in_OC`, `in_HTN`, `in_SBP`).
In Cytoscape you can draw the same pie nodes with the free **enhancedGraphics** app
(*Style → Image/Chart 1 → Pie chart*, data columns = the three `in_*` columns).

In [11]:
edges_out3 = pd.DataFrame(
    [(u, v, d["data_source"]) for u, v, d in sub3.edges(data=True)],
    columns=["source", "target", "data_source"],
)
nodes_out3 = pd.DataFrame(
    {
        "gene": list(sub3.nodes()),
        "group": [group_name(n) for n in sub3],
        "n_diseases": [len(diseases_of(n)) for n in sub3],
        **{f"in_{d}": [int(n in genes3[d]) for n in sub3] for d in DISEASES},
        **{f"score_{d}": [scores3[d].get(n) for n in sub3] for d in DISEASES},
        "degree_in_subnetwork": [sub3.degree(n) for n in sub3],
        "degree_in_full_network": [G.degree(n) for n in sub3],
    }
)

edges_out3.to_csv(OUT_DIR_3 / "cytoscape_edges.csv", index=False)
nodes_out3.to_csv(OUT_DIR_3 / "cytoscape_nodes.csv", index=False)
print(f"Saved {OUT_DIR_3 / 'cytoscape_edges.csv'} ({len(edges_out3)} links)")
print(f"Saved {OUT_DIR_3 / 'cytoscape_nodes.csv'} ({len(nodes_out3)} genes)")

Saved outputs/cutoff_0.4/network_OC_HTN_SBP/cytoscape_edges.csv (3056 links)
Saved outputs/cutoff_0.4/network_OC_HTN_SBP/cytoscape_nodes.csv (741 genes)


## Interactive HTML views

In [12]:
def pie_icon(diseases):
    """SVG pie (as a data URL) with one slice per disease, split by a thin white gap."""
    colors = [DISEASE_COLORS[d] for d in diseases]
    if len(colors) == 1:
        body = f'<circle cx="50" cy="50" r="46" fill="{colors[0]}"/>'
    else:
        body = ""
        step = 360 / len(colors)
        for i, c in enumerate(colors):
            a0, a1 = math.radians(-90 + i * step), math.radians(-90 + (i + 1) * step)
            x0, y0 = 50 + 46 * math.cos(a0), 50 + 46 * math.sin(a0)
            x1, y1 = 50 + 46 * math.cos(a1), 50 + 46 * math.sin(a1)
            large = 1 if step > 180 else 0
            body += (
                f'<path d="M50,50 L{x0:.2f},{y0:.2f} A46,46 0 {large} 1 {x1:.2f},{y1:.2f} Z" '
                f'fill="{c}" stroke="#fcfcfb" stroke-width="5" stroke-linejoin="round"/>'
            )
    svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="100" height="100" viewBox="0 0 100 100">'
        f'{body}<circle cx="50" cy="50" r="47" fill="none" stroke="#fcfcfb" stroke-width="3"/></svg>'
    )
    return "data:image/svg+xml;charset=utf-8," + quote(svg)


GROUP_ICONS = {g: pie_icon(tuple(g.replace(" only", "").split(" + "))) for g in GROUPS_3}

HTML_TEMPLATE_3 = """<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>__TITLE__</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/vis-network/9.1.9/standalone/umd/vis-network.min.js"></script>
<style>
  body { margin: 0; font-family: sans-serif; background: #fcfcfb; color: #0b0b0b; }
  #legend { position: absolute; top: 12px; left: 12px; background: #ffffffee; padding: 10px 12px;
            border: 1px solid #e0dfd9; border-radius: 6px; font-size: 14px; z-index: 1; }
  #legend label { display: flex; align-items: center; gap: 6px; margin: 3px 0; cursor: pointer; user-select: none; }
  #legend label.off { opacity: 0.35; }
  #legend input { display: none; }
  #legend img { width: 16px; height: 16px; }
  #legend small { color: #52514e; }
  #net { width: 100vw; height: 100vh; }
  div.vis-tooltip { white-space: pre-line !important; font-family: sans-serif; }
</style>
</head>
<body>
<div id="legend"><b>__TITLE__</b><br><small>click a group to hide / show it</small>__LEGEND__<small>__COUNTS__ · hover a gene for details</small></div>
<div id="net"></div>
<script>
  const nodes = new vis.DataSet(__NODES__);
  const edges = new vis.DataSet(__EDGES__);
  const network = new vis.Network(document.getElementById("net"), {nodes, edges}, {
    nodes: {shape: "image", shapeProperties: {useBorderWithImage: false, interpolation: true},
            font: {size: 22, color: "#0b0b0b", strokeWidth: 3, strokeColor: "#fcfcfb"}},
    edges: {color: {color: "#c9c8c2", highlight: "#52514e"}, width: 1, smooth: false},
    physics: {solver: "barnesHut", barnesHut: {gravitationalConstant: -40000, centralGravity: 0.4,
                                               springLength: 250, springConstant: 0.02, avoidOverlap: 1},
              stabilization: {iterations: 1000}},
    interaction: {hover: true, tooltipDelay: 100}
  });
  // once the layout has settled: stop the physics (nodes stay where they are)
  // and zoom to the main connected group (small separate pieces stay off to the side)
  network.once("stabilizationIterationsDone", () => {
    network.setOptions({physics: false});
    network.fit({nodes: __MAIN__});
  });
  // legend: click a group to hide / show its genes
  document.querySelectorAll("#legend input").forEach(box => {
    box.addEventListener("change", () => {
      box.parentElement.classList.toggle("off", !box.checked);
      nodes.update(nodes.get({filter: n => n.group_name === box.value})
                        .map(n => ({id: n.id, hidden: !box.checked})));
    });
  });
</script>
</body>
</html>"""


def draw3(graph, path, title):
    nodes = []
    for n in graph.nodes():
        ds = diseases_of(n)
        lines = [n, f"group: {group_name(n)}"]
        lines += [f"{d} score: {scores3[d].get(n):.2f}" for d in ds]
        lines.append(f"links here: {graph.degree(n)} | links in full network: {G.degree(n)}")
        nodes.append(
            {
                "id": n,
                "group_name": group_name(n),
                "label": n if len(ds) > 1 else "",
                "title": "\n".join(lines),
                "image": GROUP_ICONS[group_name(n)],
                "size": 8 + 4 * graph.degree(n) ** 0.5 + (4 if len(ds) > 1 else 0),
            }
        )
    edges = [{"from": u, "to": v} for u, v in graph.edges()]

    counts = pd.Series([group_name(n) for n in graph]).value_counts()
    legend = "".join(
        f'<label><input type="checkbox" value="{g}" checked><img src="{GROUP_ICONS[g]}">{g} ({counts.get(g, 0)})</label>'
        for g in GROUPS_3
    )
    html = (
        HTML_TEMPLATE_3.replace("__TITLE__", title)
        .replace("__LEGEND__", legend)
        .replace("__COUNTS__", f"{graph.number_of_nodes()} genes, {graph.number_of_edges()} links")
        .replace("__NODES__", json.dumps(nodes))
        .replace("__EDGES__", json.dumps(edges))
        .replace("__MAIN__", json.dumps(list(max(nx.connected_components(graph), key=len))))
    )
    with open(path, "w") as f:
        f.write(html)
    print(f"Saved {path}: {graph.number_of_nodes()} genes, {graph.number_of_edges()} links")


# View 1: all genes of the three diseases
draw3(sub3, OUT_DIR_3 / "all_genes.html", "OC + HTN + SBP disease genes")

Saved outputs/cutoff_0.4/network_OC_HTN_SBP/all_genes.html: 741 genes, 3056 links


### View 2: genes in 2+ diseases and their direct neighbours

In [13]:
multi = {n for n in sub3 if len(diseases_of(n)) > 1}
neighbours3 = {m for n in multi for m in sub3.neighbors(n)}
bridge3 = sub3.subgraph(multi | neighbours3).copy()

draw3(bridge3, OUT_DIR_3 / "shared_bridges.html", "Genes in 2+ diseases and their neighbours")

Saved outputs/cutoff_0.4/network_OC_HTN_SBP/shared_bridges.html: 467 genes, 2290 links


---
# Renin–angiotensin system (RAS) genes in the network

Where do the 23 RAS genes sit in the interactome, and how close are they to the OC / HTN / SBP disease genes?

The picture shows **every RAS gene** (pie inside a **thick dark ring**) plus every OC / HTN / SBP gene that is
**directly linked** to a RAS gene (plain pies, same colours as above). A RAS gene that is itself a disease gene
has its disease colours inside the ring; a RAS gene in no disease is grey inside the ring.
Links between two RAS genes are drawn dark, all other links light grey.

In [14]:
RAS_GENES = {  # symbol: Entrez ID
    "ACE": 1636,
    "ACE2": 59272,
    "AGT": 183,
    "AGTR1": 185,
    "AGTR2": 186,
    "REN": 5972,
    "ATP6AP2": 10159,
    "MAS1": 4142,
    "MRGPRD": 116512,
    "LNPEP": 4012,
    "ANPEP": 290,
    "ENPEP": 2028,
    "MME": 4311,
    "CMA1": 1215,
    "CTSG": 1511,
    "CPA3": 1359,
    "CTSA": 5476,
    "KLK1": 3816,
    "KLK2": 3817,
    "PRCP": 5547,
    "PREP": 5550,
    "NLN": 57486,
    "THOP1": 7064,
}
OUT_DIR_RAS = Path("outputs") / f"cutoff_{CUTOFF}" / "network_RAS"
OUT_DIR_RAS.mkdir(parents=True, exist_ok=True)

ras = {g for g in RAS_GENES if g in G}
all_disease = set().union(*genes3.values())
missing = sorted(set(RAS_GENES) - ras)
print(
    f"{len(ras)} of {len(RAS_GENES)} RAS genes are in the interactome; missing: {', '.join(missing)}"
)

ras_table = pd.DataFrame(
    [
        {
            "gene": g,
            "entrez_id": RAS_GENES[g],
            "in_interactome": g in G,
            "diseases": " + ".join(diseases_of(g)) if g in G else "",
            "links_to_RAS_genes": len(set(G[g]) & ras) if g in G else None,
            "links_to_disease_genes": len(set(G[g]) & all_disease) if g in G else None,
            "degree_in_full_network": G.degree(g) if g in G else None,
        }
        for g in RAS_GENES
    ]
)
ras_table.to_csv(OUT_DIR_RAS / "ras_genes_summary.csv", index=False)
ras_table

21 of 23 RAS genes are in the interactome; missing: CPA3, MRGPRD


,gene,entrez_id,in_interactome,diseases,links_to_RAS_genes,links_to_disease_genes,degree_in_full_network
0,ACE,1636,True,HTN + SBP,1.0,0.0,5.0
1,ACE2,59272,True,,1.0,2.0,7.0
2,AGT,183,True,HTN + SBP,9.0,2.0,17.0
3,AGTR1,185,True,HTN,2.0,9.0,25.0
4,AGTR2,186,True,,2.0,5.0,10.0
5,REN,5972,True,HTN,1.0,2.0,8.0
6,ATP6AP2,10159,True,,1.0,1.0,3.0
7,MAS1,4142,True,,1.0,1.0,1.0
8,MRGPRD,116512,False,,NaN,NaN,NaN
9,LNPEP,4012,True,,0.0,3.0,7.0


## Build the RAS subnetwork and export for Cytoscape
Node table column `is_RAS` marks the RAS genes (map it to *Border Width* in Cytoscape);
the `in_*` columns give the disease pies, exactly as in the three-disease export above.

In [15]:
ras_neighbours = {m for g in ras for m in G[g] if m in all_disease}
ras_net = G.subgraph(ras | ras_neighbours).copy()
# keep only links that touch a RAS gene (links among the neighbours themselves would hide the RAS structure)
ras_net.remove_edges_from(
    [(u, v) for u, v in list(ras_net.edges()) if u not in ras and v not in ras]
)

print(
    f"RAS view: {len(ras)} RAS genes + {len(ras_neighbours - ras)} linked disease genes, "
    f"{ras_net.number_of_edges()} links ({ras_net.subgraph(ras).number_of_edges()} between two RAS genes)"
)

pd.DataFrame(
    [(u, v, G[u][v]["data_source"], u in ras and v in ras) for u, v in ras_net.edges()],
    columns=["source", "target", "data_source", "RAS_RAS_link"],
).to_csv(OUT_DIR_RAS / "cytoscape_edges.csv", index=False)
pd.DataFrame(
    {
        "gene": list(ras_net.nodes()),
        "is_RAS": [int(n in ras) for n in ras_net],
        "group": [group_name(n) if diseases_of(n) else "no disease" for n in ras_net],
        **{f"in_{d}": [int(n in genes3[d]) for n in ras_net] for d in DISEASES},
        **{f"score_{d}": [scores3[d].get(n) for n in ras_net] for d in DISEASES},
        "degree_in_full_network": [G.degree(n) for n in ras_net],
    }
).to_csv(OUT_DIR_RAS / "cytoscape_nodes.csv", index=False)
print(f"Saved Cytoscape files to {OUT_DIR_RAS}")

RAS view: 21 RAS genes + 39 linked disease genes, 54 links (12 between two RAS genes)
Saved Cytoscape files to outputs/cutoff_0.4/network_RAS


## Interactive HTML view
Same controls as above: hover a gene for details, click a legend entry to hide / show that group.

In [16]:
from urllib.parse import unquote

RAS_RING = "#0b0b0b"
NO_DISEASE = "#c9c8c2"


def ras_icon(diseases):
    """Disease pie (grey if no disease) inside a thick dark ring that marks a RAS gene."""
    if diseases:  # reuse the pie's SVG, shrunk to fit inside the ring
        pie_svg = unquote(pie_icon(diseases).split(",", 1)[1])
        body = pie_svg.replace(
            '<svg xmlns="http://www.w3.org/2000/svg" width="100" height="100"',
            '<svg x="18" y="18" width="64" height="64"',
            1,
        )
    else:
        body = f'<circle cx="50" cy="50" r="31" fill="{NO_DISEASE}"/>'
    svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="100" height="100" viewBox="0 0 100 100">'
        f'<circle cx="50" cy="50" r="42" fill="#fcfcfb" stroke="{RAS_RING}" stroke-width="10"/>{body}</svg>'
    )
    return "data:image/svg+xml;charset=utf-8," + quote(svg)


def draw_ras(graph, path, title):
    nodes = []
    for n in graph.nodes():
        ds = diseases_of(n)
        is_ras = n in ras
        grp = "RAS gene" if is_ras else group_name(n)
        lines = [n, f"RAS gene (Entrez {RAS_GENES[n]})" if is_ras else f"group: {grp}"]
        if is_ras:
            lines.append(f"diseases: {' + '.join(ds) if ds else 'none'}")
        lines += [f"{d} score: {scores3[d].get(n):.2f}" for d in ds]
        lines.append(f"links here: {graph.degree(n)} | links in full network: {G.degree(n)}")
        nodes.append(
            {
                "id": n,
                "group_name": grp,
                "label": n if is_ras or len(ds) > 1 else "",
                "title": "\n".join(lines),
                "image": ras_icon(ds) if is_ras else GROUP_ICONS[grp],
                "size": (24 if is_ras else 12) + 3 * graph.degree(n) ** 0.5,
            }
        )
    edges = [
        {
            "from": u,
            "to": v,
            **({"color": {"color": RAS_RING}, "width": 3} if u in ras and v in ras else {}),
        }
        for u, v in graph.edges()
    ]

    counts = pd.Series([nd["group_name"] for nd in nodes]).value_counts()
    legend_groups = [("RAS gene", ras_icon(()))] + [
        (g, GROUP_ICONS[g]) for g in GROUPS_3 if counts.get(g, 0)
    ]
    legend = "".join(
        f'<label><input type="checkbox" value="{g}" checked><img src="{icon}">{g} ({counts.get(g, 0)})</label>'
        for g, icon in legend_groups
    )
    legend += (
        "<small>inside the ring: the RAS gene's own diseases (grey = none)<br>"
        f"thick dark line = link between two RAS genes<br>not in interactome: {', '.join(missing)}</small><br>"
    )
    # small graph: shorter links and a stronger pull to the centre, so unlinked RAS genes don't drift far away
    template = HTML_TEMPLATE_3.replace("centralGravity: 0.4", "centralGravity: 2.5").replace(
        "springLength: 250", "springLength: 110"
    )
    html = (
        template.replace("__TITLE__", title)
        .replace("__LEGEND__", legend)
        .replace("__COUNTS__", f"{graph.number_of_nodes()} genes, {graph.number_of_edges()} links")
        .replace("__NODES__", json.dumps(nodes))
        .replace("__EDGES__", json.dumps(edges))
        .replace("__MAIN__", json.dumps(list(graph.nodes())))
    )  # fit everything, incl. unlinked RAS genes
    with open(path, "w") as f:
        f.write(html)
    print(f"Saved {path}: {graph.number_of_nodes()} genes, {graph.number_of_edges()} links")


draw_ras(
    ras_net, OUT_DIR_RAS / "ras_network.html", "RAS genes and their linked OC / HTN / SBP genes"
)


Saved outputs/cutoff_0.4/network_RAS/ras_network.html: 60 genes, 54 links
